In [1]:
homedir = '/mnt/mirabelle/az6922_homedir/DRing/src/emp/datacentre/'
import random
import numpy as np

# from makec2s.ipynb
def genflowbytes():
    np.random.seed(0)
    
    mean_bytes = 100.0 * 1024
    shape = 1.05
    scale = mean_bytes * (shape - 1)/shape

    x = np.random.exponential(scale=1.0/shape)
    flowbytes = int(scale * np.exp(x))
    return flowbytes

def adjustbytesbymtu(flowbytes):
  mss = 1500
  return mss * ((flowbytes+mss-1)//mss)

large_flow_threshold = 10 * 1024 * 1024

(current dir: ~/DRing/src/emp/datacentre/experiments/nsdi26fall/test_dragonfly/)
python3 generate_dragonfly_topologyfiles_serverfiles.py
python3 generate_suk_netpathfiles.py

In [11]:
p=2
a=3
h=1
stime = 10
seed = 1
cmfile = 'experiments/nsdi26fall/test_dragonfly/cmfiles/singleflow.cm'
topologytype = 3
npfile = f'experiments/nsdi26fall/test_dragonfly/npfiles/df_p{p}_a{a}_h{h}_su2.np'
pwfileprefix = f'experiments/nsdi26fall/test_dragonfly/pwfiles/pathweight_df_p{p}_a{a}_h{h}_su2_singleflow.var'
serverfile = f'experiments/nsdi26fall/test_dragonfly/svfiles/df_p{p}_a{a}_h{h}.sv'
topologyfile = f'experiments/nsdi26fall/test_dragonfly/tpfiles/df_p{p}_a{a}_h{h}.edgelist'
outfile = f'experiments/nsdi26fall/test_dragonfly/outfiles/df_p{p}_a{a}_h{h}_su2_singleflow.out'

In [7]:
# current dir: ~/DRing/src/emp/datacentre/
nswitches = a*(a*h+1)
nhosts = a*p*(a*h+1)
flowstart = 0
flowend = 10
varfile = f'experiments/nsdi26fall/test_dragonfly/pwfiles/pathtraffic_df_p{p}_a{a}_h{h}_su2_singleflow.var'
qvarfile = f'experiments/nsdi26fall/test_dragonfly/pwfiles/pathweight_df_p{p}_a{a}_h{h}_su2_singleflow.var'
print(f"python3 generate_pathweightfiles.py --graphfile {topologyfile} --serverfile {serverfile} --numsw {nswitches} --numserver {nhosts} --netpathfile {npfile} --flowfile {cmfile} --flowstart {flowstart} --flowend {flowend} --numfaillink 0 --linkfailurefile none --varfile {varfile} --qvarfile {qvarfile}")

python3 generate_pathweightfiles.py --graphfile experiments/nsdi26fall/test_dragonfly/tpfiles/df_p2_a3_h1.edgelist --serverfile experiments/nsdi26fall/test_dragonfly/svfiles/df_p2_a3_h1.sv --numsw 12 --numserver 24 --netpathfile experiments/nsdi26fall/test_dragonfly/npfiles/df_p2_a3_h1_su2.np --flowfile experiments/nsdi26fall/test_dragonfly/cmfiles/singleflow.cm --flowstart 0 --flowend 10 --numfaillink 0 --linkfailurefile none --varfile experiments/nsdi26fall/test_dragonfly/pwfiles/pathtraffic_df_p2_a3_h1_su2_singleflow.var --qvarfile experiments/nsdi26fall/test_dragonfly/pwfiles/pathweight_df_p2_a3_h1_su2_singleflow.var


In [14]:
# generate conf file
print(f"./eval -stime {stime} -seed {seed} -df_p {p} -df_a {a} -df_h {h} -cmfile {cmfile} -topologytype {topologytype} -npfile {npfile} -pwfileprefix {pwfileprefix} -numintervals 1 -serverfile {serverfile} -topologyfile {topologyfile} > {outfile}")

./eval -stime 10 -seed 1 -df_p 2 -df_a 3 -df_h 1 -cmfile experiments/nsdi26fall/test_dragonfly/cmfiles/singleflow.cm -topologytype 3 -npfile experiments/nsdi26fall/test_dragonfly/npfiles/df_p2_a3_h1_su2.np -pwfileprefix experiments/nsdi26fall/test_dragonfly/pwfiles/pathweight_df_p2_a3_h1_su2_singleflow.var -numintervals 1 -serverfile experiments/nsdi26fall/test_dragonfly/svfiles/df_p2_a3_h1.sv -topologyfile experiments/nsdi26fall/test_dragonfly/tpfiles/df_p2_a3_h1.edgelist > experiments/nsdi26fall/test_dragonfly/outfiles/df_p2_a3_h1_su2_singleflow.out


In [3]:
# generate connection_matrices file (1)
unv1bytes = 0
unv1file = f'{homedir}rawtrafficfiles/unv1'
maxinterval = 0
with open(unv1file, 'r') as f:
    lines = f.readlines()
    for line in lines:
        tokens = line.split(',')
        # 0,32,31,10500
        # interval,fromserver,toserver,bytes
        unv1bytes += int(tokens[3])
        maxinterval = max(maxinterval, int(tokens[0]))
print(f'unv1bytes {unv1bytes}, maxinterval {maxinterval}, fullload {bw * stime * nlinks / 1000}, ratio {(bw * stime * nlinks / 1000) / unv1bytes}')

unv1bytes 162036861000, maxinterval 7, fullload 412058769408.0, ratio 2.5429940253409375


In [5]:
# generate connection_matrices file (2)
random.seed(0)
for load in load_list:
    totalbytes = bw * stime / 1000 * nlinks * load / 100  # B
    mult = totalbytes / unv1bytes
    actualbytes = 0
    cmfile = f'cmfiles/dring_load{load}.cm'
    with open(cmfile, 'w') as fw:
        with open(unv1file, 'r') as fr:
            lines = fr.readlines()
            iline = 0
            while actualbytes < totalbytes:
                line = lines[iline]
                tokens = line.split(',')
                interval = int(tokens[0])
                fromserver = int(tokens[1])
                toserver = int(tokens[2])
                multbytes = int(tokens[3])

                if fromserver >= nhosts or toserver >= nhosts:
                    iline += 1
                    if iline >= len(lines):
                        iline = 0
                        if mult-1>0:
                            mult = mult-1
                    continue

                if mult >= 1 or (random.random() < mult):
                    multbytes = adjustbytesbymtu(multbytes)
    
                    # generate random start time
                    start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

                    fw.write(f'{fromserver},{toserver},{int(multbytes)},{start_time_ms:.4f}\n')
                    actualbytes += int(multbytes)

                iline += 1
                if iline >= len(lines):
                    iline = 0
                    if mult-1>0:
                        mult = mult-1

                    # print(f'actualbytes {actualbytes}, totalbytes {totalbytes}, mult {mult}', end='\r')

    print(f'load {load}%, totalbytes {totalbytes}, unv1bytes {unv1bytes}, mult {mult}, actualbytes {actualbytes}')


load 1%, totalbytes 4120587694.08, unv1bytes 162036861000, mult 0.025429940253409375, actualbytes 4122780000
load 2%, totalbytes 8241175388.16, unv1bytes 162036861000, mult 0.05085988050681875, actualbytes 8247067500
load 3%, totalbytes 12361763082.24, unv1bytes 162036861000, mult 0.07628982076022812, actualbytes 12361767000
load 4%, totalbytes 16482350776.32, unv1bytes 162036861000, mult 0.1017197610136375, actualbytes 16482409500
load 5%, totalbytes 20602938470.4, unv1bytes 162036861000, mult 0.1271497012670469, actualbytes 20603155500
load 6%, totalbytes 24723526164.48, unv1bytes 162036861000, mult 0.15257964152045625, actualbytes 24723550500
load 7%, totalbytes 28844113858.56, unv1bytes 162036861000, mult 0.17800958177386564, actualbytes 28844121000
load 8%, totalbytes 32964701552.64, unv1bytes 162036861000, mult 0.203439522027275, actualbytes 32964703500
load 9%, totalbytes 37085289246.72, unv1bytes 162036861000, mult 0.2288694622806844, actualbytes 37085295000


In [6]:
# generate pathweight file (1)
interval_stime = stime / nintervals
with open('dringsu2_generate_pwfiles.conf', 'w') as f:
    for load in load_list:
        cmfile = f'cmfiles/dring_load{load}.cm'
        for interval in range(nintervals):
            flowstart = interval_stime * interval
            flowend = interval_stime * (interval + 1)
            varfile = f'{homedir}rawpathweightfiles/pathtraffic_dring_{nhosts}_{nswitches}_{k}_su2_unv1_load{load}_interval{interval}.var'
            qvarfile = f'{homedir}rawpathweightfiles/pathweight_dring_{nhosts}_{nswitches}_{k}_su2_unv1_load{load}_interval{interval}.var'
            f.write(f"python3 {homedir}generate_pathweightfiles.py --graphfile {homedir}{topologyfile} --serverfile {homedir}{serverfile} --numsw {nswitches} --numserver {nhosts} --netpathfile {homedir}{npfile} --flowfile {cmfile} --flowstart {flowstart} --flowend {flowend} --numfaillink 0 --linkfailurefile none --varfile {varfile} --qvarfile {qvarfile}\n")

(current dir: ~/DRing/src/emp/datacentre/experiments/nsdi26fall/eval_main/unv1/)
python3 ../../../../pararun.py --conf dringsu2_generate_pwfiles.conf --worker 72

In [7]:
# generate pathweight file (2)
intervaldict = {0:0,1:0,2:1,3:2,4:3,5:4,6:5,7:6} # to:from
with open('dringsu2_copy_pwfiles.conf', 'w') as f:
    for load in load_list:
        for interval in range(nintervals):
            fromfile = f'{homedir}rawpathweightfiles/pathweight_dring_{nhosts}_{nswitches}_{k}_su2_unv1_load{load}_interval{intervaldict[interval]}.var'
            tofile = f'{homedir}experiments/nsdi26fall/eval_main/unv1/pwfiles/pathweight_dring_su2_unv1_load{load}_interval{interval}.pw'
            f.write(f'cp {fromfile} {tofile}\n')

actually run the copy commands in datacentre/

In [8]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/eval_main/unv1/run_dringsu2.conf'
with open(conffile, 'w') as f:
    for seed in seed_list:
        for load in load_list:
            cmfile = f'experiments/nsdi26fall/eval_main/unv1/cmfiles/dring_load{load}.cm'
            pwfileprefix = f'experiments/nsdi26fall/eval_main/unv1/pwfiles/pathweight_dring_su2_unv1_load{load}_interval'
            outfile = f'experiments/nsdi26fall/eval_main/unv1/outfiles/dringsu2_load{load}_seed{seed}.out'
            f.write(f"./eval -stime {stime} -seed {seed} -cmfile {cmfile} -topologytype {topologytype} -numswitches {nswitches} -numhosts {nhosts} -os {os} -ls_k {k} -npfile {npfile} -pwfileprefix {pwfileprefix} -numintervals {nintervals} -serverfile {serverfile} -topologyfile {topologyfile} > {outfile}\n")
            

python3 pararun.py --conf experiments/nsdi26fall/eval_main/unv1/run_dringsu2.conf --worker 32